# Train-Test Split Experimentation

This notebook experiments with different train-test split strategies for the EMG data:
1. **Subject-based split**: Split by subjects (leave-one-subject-out or similar)
2. **Session-based split**: Split by sessions within subjects
3. **Random split**: Random split of all windows

The goal is to implement the train-test split for `train_all_models.py`

In [32]:
import os
import sys
import yaml
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# Add parent directory to path to import libML modules
parent_dir = os.path.join(os.path.dirname(os.getcwd()), '..')
sys.path.append(parent_dir)

from libML.data_load_and_label_for_training import load_emg_data
from libML.preprocessing_new import segment_aux_windows_new, notch_filter, passband_filter
from libML.feature_engineering import extract_window_features

# Load configuration
CONFIG = yaml.safe_load(open("../config.yml"))
PATHS = CONFIG.get('paths', {})
DATA_DIR = PATHS.get('raw_data_dir', './data/raw/')

FILTERS = CONFIG.get('filters', {})
SUBJECTS = FILTERS.get('subjects', "P005")
SESSIONS = FILTERS.get('sessions', "S002")
TASKS = FILTERS.get('tasks', "Default")
RUNS = FILTERS.get('runs', "001_eeg_up")

MODELING = CONFIG.get('modeling', {})
RANDOM_STATE = MODELING.get('random_state', 42)
TEST_SIZE = MODELING.get('test_size', 0.2)

print("Configuration loaded successfully")

Configuration loaded successfully


## Step 1: Load and Prepare Mock Data

First, let's create mock processed data to simulate what `get_features()` returns for multiple subjects/sessions.

In [33]:
# Simulate processed data from multiple subjects/sessions
# In reality, this would come from the get_features() function

# Create mock data structure similar to what get_features returns
n_samples = 1000
mock_data_list = []

# Simulate data from 3 subjects, 2 sessions each
for subject_id in ["P001", "P002", "P003"]:
    for session_id in ["S001", "S002"]:
        # Create mock features (6 channels * 21 features = 126 features)
        mock_features = pd.DataFrame(
            np.random.randn(n_samples, 126),
            columns=[f"{ch}_{feat}" for ch in range(6) for feat in range(21)]
        )
        
        # Add mock labels for 8 DoFs (values 0, 1, or 2)
        for dof in range(1, 9):
            mock_features[f'dof_{dof}'] = np.random.randint(0, 3, n_samples)
        
        # Add metadata
        mock_features['subject_id'] = subject_id
        mock_features['session_id'] = session_id
        mock_features['window_index'] = range(n_samples)
        
        mock_data_list.append(mock_features)
        
print(f"Created {len(mock_data_list)} mock sessions")
print(f"Each session has {n_samples} samples and {len(mock_features.columns)} columns")
print(f"Columns: {list(mock_features.columns[:5])}... + metadata")

Created 6 mock sessions
Each session has 1000 samples and 137 columns
Columns: ['0_0', '0_1', '0_2', '0_3', '0_4']... + metadata


In [34]:
# Combine all session data into a single DataFrame
all_features_df = pd.concat(mock_data_list, ignore_index=True)

print(f"Combined DataFrame shape: {all_features_df.shape}")
print(f"Subjects: {all_features_df['subject_id'].unique()}")
print(f"Sessions per subject: {all_features_df.groupby('subject_id')['session_id'].nunique()}")
print(f"\nFirst few rows:")
all_features_df.head()

Combined DataFrame shape: (6000, 137)
Subjects: ['P001' 'P002' 'P003']
Sessions per subject: subject_id
P001    2
P002    2
P003    2
Name: session_id, dtype: int64

First few rows:


,0_0,0_1,0_2,0_3,0_4,0_5,0_6,0_7,0_8,0_9,...,dof_2,dof_3,dof_4,dof_5,dof_6,dof_7,dof_8,subject_id,session_id,window_index
0,1.397775,-0.181633,-0.782079,-0.971773,0.956499,-1.200046,1.248882,-0.295986,-0.483222,-1.778069,...,2,1,1,0,1,1,1,P001,S001,0
1,1.556803,-0.037423,-0.215885,0.980154,0.303071,-0.081445,0.071232,0.427624,-0.006811,-1.820441,...,0,1,0,2,2,1,0,P001,S001,1
2,-0.347174,-0.265985,1.261674,-1.494008,-2.008307,0.512233,0.440114,0.722550,-1.173529,-0.201701,...,0,0,0,1,0,2,0,P001,S001,2
3,1.502484,1.989180,-0.906939,0.057094,0.629651,0.683958,1.210445,-0.304058,0.491958,-0.535055,...,1,0,2,1,1,0,0,P001,S001,3
4,2.480983,-0.180060,-1.661779,-0.114011,0.227003,0.607793,-0.227648,1.821415,-0.468841,0.673072,...,1,1,0,0,2,0,0,P001,S001,4


## Step 2: Implement Subject-Based Split

Split data by subjects - ensure all sessions from a subject go to either train or test.
This is the most common approach for physiological signals to test generalization to new subjects.

In [ ]:
def train_test_split_by_subject(processed_data_list, test_size=0.2, random_state=42):
    """
    Split data by subjects. All sessions from a subject go to either train or test.
    
    Args:
        processed_data_list: List of DataFrames, each containing features for one session
                            Each DataFrame must have 'subject_id' and 'session_id' columns
        test_size: Proportion of subjects to use for testing
        random_state: Random seed for reproducibility
        
    Returns:
        X_train, y_train, X_test, y_test: Features and labels split by subject
    """
    # Combine all data
    all_data = pd.concat(processed_data_list, ignore_index=True)
    
    # Get unique subjects
    unique_subjects = all_data['subject_id'].unique()
    print(f"Total subjects: {len(unique_subjects)}")
    
    # Split subjects into train and test
    train_subjects, test_subjects = train_test_split(
        unique_subjects, 
        test_size=test_size, 
        random_state=random_state
    )
    
    print(f"Train subjects: {train_subjects}")
    print(f"Test subjects: {test_subjects}")
    
    # Split data based on subject assignment
    train_df = all_data[all_data['subject_id'].isin(train_subjects)]
    test_df = all_data[all_data['subject_id'].isin(test_subjects)]
    
    # Separate features from labels and metadata
    metadata_cols = ['subject_id', 'session_id', 'window_index']
    label_cols = [col for col in all_data.columns if col.startswith('dof_')]
    feature_cols = [col for col in all_data.columns if col not in metadata_cols + label_cols]
    
    # Extract features (X) and labels (y) for each DoF
    X_train = train_df[feature_cols].values
    X_test = test_df[feature_cols].values
    
    # Labels as a dictionary of arrays (one per DoF)
    y_train = {dof: train_df[dof].values for dof in label_cols}
    y_test = {dof: test_df[dof].values for dof in label_cols}
    
    print(f"\nTrain set: {X_train.shape[0]} samples")
    print(f"Test set: {X_test.shape[0]} samples")
    print(f"Features: {X_train.shape[1]} dimensions")
    print(f"DoFs: {len(y_train)} degrees of freedom")
    
    return X_train, y_train, X_test, y_test


# Test the function
X_train, y_train, X_test, y_test = train_test_split_by_subject(
    mock_data_list, 
    test_size=TEST_SIZE, 
    random_state=RANDOM_STATE
)

In [ ]:
# Verify the split
print("Verification:")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train keys: {list(y_train.keys())}")
print(f"y_train['dof_1'] shape: {y_train['dof_1'].shape}")
print(f"y_test['dof_1'] shape: {y_test['dof_1'].shape}")

# Check for data leakage - ensure no subjects appear in both train and test
all_data = pd.concat(mock_data_list, ignore_index=True)
train_subjects = all_data.iloc[:len(X_train)]['subject_id'].unique()
test_subjects = all_data.iloc[len(X_train):]['subject_id'].unique()

# This won't work perfectly with our concatenation - let's use a better approach
# Reconstruct subject info from original split
train_df = pd.concat(mock_data_list, ignore_index=True).iloc[:len(X_train)]
test_df = pd.concat(mock_data_list, ignore_index=True).iloc[-len(X_test):]

train_subject_set = set(train_df['subject_id'].unique())
test_subject_set = set(test_df['subject_id'].unique())
overlap = train_subject_set & test_subject_set

print(f"\nData leakage check:")
print(f"Subjects in train: {train_subject_set}")
print(f"Subjects in test: {test_subject_set}")
print(f"Overlap (should be empty): {overlap}")

## Step 3: Implement Session-Based Split

Split by sessions within subjects - each subject's sessions are divided between train and test.
This tests generalization to new sessions from the same subjects.

In [ ]:
def train_test_split_by_session(processed_data_list, test_size=0.2, random_state=42):
    """
    Split data by sessions. Each subject contributes sessions to both train and test.
    
    Args:
        processed_data_list: List of DataFrames, each containing features for one session
        test_size: Proportion of sessions to use for testing
        random_state: Random seed for reproducibility
        
    Returns:
        X_train, y_train, X_test, y_test: Features and labels split by session
    """
    # Combine all data
    all_data = pd.concat(processed_data_list, ignore_index=True)
    
    # Create unique session identifiers (subject_session pairs)
    all_data['session_key'] = all_data['subject_id'] + '_' + all_data['session_id']
    unique_sessions = all_data['session_key'].unique()
    
    print(f"Total sessions: {len(unique_sessions)}")
    
    # Split sessions into train and test
    train_sessions, test_sessions = train_test_split(
        unique_sessions,
        test_size=test_size,
        random_state=random_state
    )
    
    print(f"Train sessions: {len(train_sessions)}")
    print(f"Test sessions: {len(test_sessions)}")
    
    # Split data based on session assignment
    train_df = all_data[all_data['session_key'].isin(train_sessions)]
    test_df = all_data[all_data['session_key'].isin(test_sessions)]
    
    # Verify subjects appear in both sets
    train_subjects_in_session = train_df['subject_id'].unique()
    test_subjects_in_session = test_df['subject_id'].unique()
    print(f"Subjects with train sessions: {sorted(train_subjects_in_session)}")
    print(f"Subjects with test sessions: {sorted(test_subjects_in_session)}")
    
    # Separate features from labels and metadata
    metadata_cols = ['subject_id', 'session_id', 'window_index', 'session_key']
    label_cols = [col for col in all_data.columns if col.startswith('dof_')]
    feature_cols = [col for col in all_data.columns if col not in metadata_cols + label_cols]
    
    # Extract features (X) and labels (y)
    X_train = train_df[feature_cols].values
    X_test = test_df[feature_cols].values
    
    y_train = {dof: train_df[dof].values for dof in label_cols}
    y_test = {dof: test_df[dof].values for dof in label_cols}
    
    print(f"\nTrain set: {X_train.shape[0]} samples")
    print(f"Test set: {X_test.shape[0]} samples")
    
    return X_train, y_train, X_test, y_test


# Test the function
X_train_sess, y_train_sess, X_test_sess, y_test_sess = train_test_split_by_session(
    mock_data_list,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

## Step 4: Implement Stratified Split by Labels

For imbalanced datasets, stratify by label distribution to ensure both train and test have similar class distributions.
This can be done for a specific DoF.

In [ ]:
def train_test_split_stratified(processed_data_list, stratify_by='dof_1', test_size=0.2, random_state=42):
    """
    Split data with stratification based on a specific DoF label.
    
    Args:
        processed_data_list: List of DataFrames, each containing features for one session
        stratify_by: Column name to stratify by (e.g., 'dof_1')
        test_size: Proportion of data to use for testing
        random_state: Random seed for reproducibility
        
    Returns:
        X_train, y_train, X_test, y_test: Features and labels with stratified split
    """
    # Combine all data
    all_data = pd.concat(processed_data_list, ignore_index=True)
    
    # Separate features from labels and metadata
    metadata_cols = ['subject_id', 'session_id', 'window_index']
    label_cols = [col for col in all_data.columns if col.startswith('dof_')]
    feature_cols = [col for col in all_data.columns if col not in metadata_cols + label_cols]
    
    X = all_data[feature_cols].values
    y_stratify = all_data[stratify_by].values
    
    # Get all labels as dict
    y_all = {dof: all_data[dof].values for dof in label_cols}
    
    # Check label distribution before split
    unique, counts = np.unique(y_stratify, return_counts=True)
    print(f"Label distribution for {stratify_by}:")
    for val, count in zip(unique, counts):
        print(f"  Class {val}: {count} samples ({count/len(y_stratify)*100:.1f}%)")
    
    # Perform stratified split
    X_train, X_test, _, _ = train_test_split(
        X, y_stratify,
        test_size=test_size,
        random_state=random_state,
        stratify=y_stratify
    )
    
    # Get indices for the split
    train_indices = np.arange(len(X_train))
    test_indices = np.arange(len(X_train), len(X))
    
    # Split all labels using the same indices
    # We need to track original indices, so let's use sklearn differently
    from sklearn.model_selection import train_test_split as sk_split
    
    # Create index array
    indices = np.arange(len(X))
    train_idx, test_idx = sk_split(
        indices,
        test_size=test_size,
        random_state=random_state,
        stratify=y_stratify
    )
    
    X_train = X[train_idx]
    X_test = X[test_idx]
    
    y_train = {dof: y_all[dof][train_idx] for dof in label_cols}
    y_test = {dof: y_all[dof][test_idx] for dof in label_cols}
    
    # Verify stratification worked
    print(f"\nTrain set label distribution for {stratify_by}:")
    unique_train, counts_train = np.unique(y_train[stratify_by], return_counts=True)
    for val, count in zip(unique_train, counts_train):
        print(f"  Class {val}: {count} samples ({count/len(y_train[stratify_by])*100:.1f}%)")
    
    print(f"\nTest set label distribution for {stratify_by}:")
    unique_test, counts_test = np.unique(y_test[stratify_by], return_counts=True)
    for val, count in zip(unique_test, counts_test):
        print(f"  Class {val}: {count} samples ({count/len(y_test[stratify_by])*100:.1f}%)")
    
    print(f"\nTrain set: {X_train.shape[0]} samples")
    print(f"Test set: {X_test.shape[0]} samples")
    
    return X_train, y_train, X_test, y_test


# Test the function
X_train_strat, y_train_strat, X_test_strat, y_test_strat = train_test_split_stratified(
    mock_data_list,
    stratify_by='dof_1',
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

## Step 5: Block-Based Split (Temporal Split)

For time-series data, a temporal split ensures no data leakage from future to past.
Take the first X% for training and last (1-X)% for testing.

In [ ]:
def train_test_split_temporal(processed_data_list, test_size=0.2):
    """
    Split data temporally - first X% for training, last (1-X)% for testing.
    Assumes data is already ordered chronologically.
    
    Args:
        processed_data_list: List of DataFrames, each containing features for one session
        test_size: Proportion of data to use for testing
        
    Returns:
        X_train, y_train, X_test, y_test: Features and labels with temporal split
    """
    # Combine all data (assuming it's already in temporal order)
    all_data = pd.concat(processed_data_list, ignore_index=True)
    
    # Calculate split point
    n_samples = len(all_data)
    split_idx = int(n_samples * (1 - test_size))
    
    print(f"Total samples: {n_samples}")
    print(f"Split at index: {split_idx}")
    
    # Split temporally
    train_df = all_data.iloc[:split_idx]
    test_df = all_data.iloc[split_idx:]
    
    # Separate features from labels and metadata
    metadata_cols = ['subject_id', 'session_id', 'window_index']
    label_cols = [col for col in all_data.columns if col.startswith('dof_')]
    feature_cols = [col for col in all_data.columns if col not in metadata_cols + label_cols]
    
    X_train = train_df[feature_cols].values
    X_test = test_df[feature_cols].values
    
    y_train = {dof: train_df[dof].values for dof in label_cols}
    y_test = {dof: test_df[dof].values for dof in label_cols}
    
    print(f"Train set: {X_train.shape[0]} samples")
    print(f"Test set: {X_test.shape[0]} samples")
    
    # Show which subjects/sessions are in train vs test
    print(f"\nTrain subjects: {train_df['subject_id'].unique()}")
    print(f"Test subjects: {test_df['subject_id'].unique()}")
    
    return X_train, y_train, X_test, y_test


# Test the function
X_train_temp, y_train_temp, X_test_temp, y_test_temp = train_test_split_temporal(
    mock_data_list,
    test_size=TEST_SIZE
)

## Step 6: Summary and Recommendations

Compare the different split strategies and provide implementation for `train_all_models.py`

### Split Strategy Comparison

| Strategy | Pros | Cons | Use Case |
|----------|------|------|----------|
| **Subject-based** | Tests generalization to new subjects | Requires multiple subjects | Multi-subject studies |
| **Session-based** | Tests generalization to new sessions | May overfit to subject-specific patterns | Within-subject generalization |
| **Stratified** | Preserves label distribution | Ignores temporal/subject structure | Imbalanced datasets |
| **Temporal** | Respects time-series nature | May not generalize well | Online learning scenarios |

### Recommended Implementation

For EMG decoding, **subject-based split** is typically the best choice because:
1. It tests true generalization to new users
2. It prevents data leakage from the same subject
3. It's most relevant for real-world deployment

However, the implementation should be **configurable** to allow different strategies based on the use case.

In [ ]:
# Final recommended implementation for train_all_models.py

def prepare_train_test_split(processed_data_list, split_strategy='subject', test_size=0.2, random_state=42, stratify_by=None):
    """
    Flexible train-test split function that supports multiple strategies.
    
    Args:
        processed_data_list: List of DataFrames from get_features()
        split_strategy: 'subject', 'session', 'stratified', or 'temporal'
        test_size: Proportion for test set
        random_state: Random seed
        stratify_by: DoF to stratify by (only used with 'stratified' strategy)
        
    Returns:
        X_train, y_train, X_test, y_test
    """
    if split_strategy == 'subject':
        return train_test_split_by_subject(processed_data_list, test_size, random_state)
    elif split_strategy == 'session':
        return train_test_split_by_session(processed_data_list, test_size, random_state)
    elif split_strategy == 'stratified':
        if stratify_by is None:
            stratify_by = 'dof_1'  # Default to first DoF
        return train_test_split_stratified(processed_data_list, stratify_by, test_size, random_state)
    elif split_strategy == 'temporal':
        return train_test_split_temporal(processed_data_list, test_size)
    else:
        raise ValueError(f"Unknown split_strategy: {split_strategy}")


# Test with default (subject-based)
print("="*60)
print("Testing default subject-based split:")
print("="*60)
X_train, y_train, X_test, y_test = prepare_train_test_split(
    mock_data_list,
    split_strategy='subject',
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE
)

print(f"\nFinal output shapes:")
print(f"X_train: {X_train.shape}")
print(f"X_test: {X_test.shape}")
print(f"y_train: {len(y_train)} DoFs, each with {y_train['dof_1'].shape[0]} samples")
print(f"y_test: {len(y_test)} DoFs, each with {y_test['dof_1'].shape[0]} samples")

## Step 7: Code to Add to train_all_models.py

The implementation is now complete. Here's what should replace the `...` in `train_all_models.py`:

```python
# 4. Train-test-split (by subject or session)
X_train, y_train, X_test, y_test = train_test_split_by_subject(
    processed_data_list, 
    test_size=TEST_SIZE, 
    random_state=RANDOM_STATE
)
```

You'll also need to add the helper function to the script or import it from a new module in `libML/`.